# Módulo 4 · Clase 7 (Teoría) — Generative AI, modelos multimodales y agentes
### Deep Learning · Apunte de cátedra · Cierre del curso

Esta clase conecta todo lo aprendido (MLP, CNN, Transformers) con la frontera del campo: cómo las máquinas **generan** contenido, **combinan modalidades** y **actúan** en el mundo.

**Objetivos.** Al terminar, los estudiantes deberían poder:
1. Comparar los principales modelos generativos: autoencoders, VAE, GAN y difusión.
2. Explicar cómo funcionan los modelos de difusión y Stable Diffusion.
3. Entender el aprendizaje multimodal: CLIP, BLIP-2 y los LLM con visión (LLaVA).
4. Describir qué es un agente basado en LLM: uso de herramientas, ReAct y RAG.

**Agenda (≈ 3 horas, con un descanso):**
| Bloque | Tema | ~min |
|---|---|---|
| 0 | Panorama: ¿qué es la IA generativa? | 10 |
| 1 | Modelos generativos: AE, VAE, GAN | 30 |
| 2 | Modelos de difusión y Stable Diffusion | 35 |
| — | *Descanso* | 15 |
| 3 | Modelos multimodales: CLIP, BLIP, LLaVA | 35 |
| 4 | Agentes: tool use, ReAct y RAG | 35 |
| 5 | Cierre del curso | 10 |


In [ ]:
# Setup
import numpy as np, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0); np.random.seed(0)
print("PyTorch:", torch.__version__, "| GPU:", torch.cuda.is_available())

## Bloque 0 · Panorama: ¿qué es la IA generativa?

Hasta ahora vimos sobre todo modelos **discriminativos**: dado $x$, predicen una etiqueta $y$ (clasificar una imagen, un sentimiento). Los modelos **generativos** aprenden la *distribución* de los datos para **crear muestras nuevas**: imágenes, texto, audio.

Formalmente, en vez de modelar $p(y \mid x)$, modelan $p(x)$ (o $p(x \mid \text{condición})$). Veremos cuatro familias —autoencoders, VAE, GAN y difusión— y luego cómo se combinan con lenguaje (multimodal) y con acción (agentes).

## Bloque 1 · Modelos generativos

### Autoencoder
Un **autoencoder** comprime la entrada a un código latente (encoder) y la reconstruye (decoder). Aprende una representación compacta, pero su espacio latente no está organizado para *generar*: muestrear un punto al azar rara vez produce algo válido.

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/m4_00_autoencoder.png" width="760">

<sub>Autoencoder: encoder → código latente (cuello de botella) → decoder. La salida intenta reconstruir la entrada.</sub>

### Variational Autoencoder (VAE) [Ver infografía](https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/infografias/M4/01_VAE.png)
El **VAE** hace el espacio latente **continuo y muestreable**: el encoder produce una *media* $\mu$ y una *desviación* $\sigma$, y se muestrea $z \sim \mathcal{N}(\mu, \sigma)$. Una regularización (divergencia KL) ordena el espacio latente para que puntos cercanos generen salidas parecidas. Así, muestrear $z$ al azar **sí** produce muestras nuevas y coherentes.

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/m4_01_vae.png" width="760">

<sub>VAE: el encoder produce μ y σ; se muestrea un vector latente z y el decoder genera la salida.</sub>

In [ ]:
# Demo: el 'reparameterization trick' del VAE
# z = mu + sigma * eps   (eps ~ N(0,1)) -> permite muestrear de forma diferenciable
mu    = torch.tensor([0.0, 2.0])
sigma = torch.tensor([1.0, 0.5])
muestras = torch.stack([mu + sigma*torch.randn(2) for _ in range(500)])
plt.figure(figsize=(4,4))
plt.scatter(muestras[:,0], muestras[:,1], s=8, alpha=0.4)
plt.scatter(*mu, color='red', s=80, label='μ (centro)')
plt.title("Muestreo del espacio latente del VAE"); plt.legend(); plt.grid(True); plt.show()

### Generative Adversarial Network (GAN) [Ver infografía](https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/infografias/M4/02_GAN.png)
Una **GAN** enfrenta dos redes: un **generador** $G$ crea muestras falsas a partir de ruido; un **discriminador** $D$ intenta distinguir reales de falsas. Compiten en un juego *minimax*: $G$ mejora hasta engañar a $D$. Producen imágenes muy nítidas, pero el entrenamiento es inestable.

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/m4_02_gan.png" width="760">

<sub>GAN: el generador crea muestras desde ruido; el discriminador las distingue de las reales. Ambos compiten y mejoran.</sub>

$$ \min_G \max_D \; \mathbb{E}_{x\sim \text{datos}}[\log D(x)] + \mathbb{E}_{z\sim \text{ruido}}[\log(1 - D(G(z)))] $$

**Resumen comparativo:** el VAE es estable pero genera imágenes algo borrosas; la GAN genera muy nítido pero entrena con dificultad; la **difusión** (que viene ahora) combina alta calidad con entrenamiento estable, y domina la generación de imágenes hoy.

## Bloque 2 · Modelos de difusión [Ver infografía](https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/infografias/M4/03_difusion.png)

La idea es elegante. **Proceso hacia adelante (forward):** tomamos una imagen y le agregamos ruido gaussiano poco a poco, en muchos pasos, hasta convertirla en ruido puro. **Proceso inverso (reverse):** entrenamos una red (típicamente una **U-Net**) para *quitar* ese ruido paso a paso. Para generar, partimos de ruido puro y aplicamos el proceso inverso aprendido.

In [ ]:
# Demo: el proceso forward de difusión (agregar ruido progresivamente)
img0 = np.zeros((64,64), np.float32)
img0[16:48,16:48] = 1.0; img0[24:40,24:40] = 0.4   # figura simple
betas = np.linspace(0, 1, 6)
fig, ax = plt.subplots(1, 6, figsize=(13,2.4))
for i, b in enumerate(betas):
    ruidosa = np.sqrt(1-b)*img0 + np.sqrt(b)*np.random.randn(64,64)
    ax[i].imshow(ruidosa, cmap='gray'); ax[i].set_title(f"t={i}"); ax[i].axis('off')
plt.suptitle("Proceso forward: la imagen se convierte en ruido. La red aprende a invertirlo.")
plt.show()

### Latent Diffusion y Stable Diffusion
Difundir en el espacio de píxeles es costoso. **Latent Diffusion** hace la difusión en un **espacio latente comprimido** (obtenido con un VAE), mucho más barato.

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/m4_03_latent_diffusion.png" width="620">

<sub>Latent Diffusion: el proceso de difusión ocurre en un espacio latente comprimido (vía un autoencoder), no sobre los píxeles.</sub>

**Stable Diffusion** es un modelo de difusión latente *condicionado por texto*. Tiene tres piezas: un **VAE** (comprime/descomprime imágenes), una **U-Net** que quita ruido en el latente, y un **codificador de texto** (tipo CLIP) que convierte el prompt en una guía. La U-Net usa atención cruzada para condicionar la generación al texto.

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/m4_04_stable_diffusion.png" width="760">

<sub>Stable Diffusion: VAE (espacio latente) + U-Net denoiser + condicionamiento textual (CLIP) mediante atención cruzada.</sub>

## Bloque 3 · Modelos multimodales

Los modelos **multimodales** combinan más de un tipo de dato (imagen + texto, audio, etc.). La clave es proyectar las distintas modalidades a un **espacio compartido**.

### CLIP: aprendizaje contrastivo imagen-texto [Ver infografía](https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/infografias/M4/04_CLIP.png)
**CLIP** entrena dos encoders (uno de imagen, uno de texto) para que, dado un par (imagen, descripción) correcto, sus vectores sean **cercanos**, y lejanos para pares incorrectos. Con un *batch* de $N$ pares, se maximiza la similitud en la diagonal de la matriz $N\times N$ y se minimiza fuera de ella.

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/m4_05_clip.png" width="760">

<sub>CLIP: encoders de imagen y texto entrenados con pérdida contrastiva para alinear pares (imagen, texto) en un espacio compartido.</sub>

El resultado es poderosísimo: permite **clasificación zero-shot**. Para clasificar una imagen entre categorías nuevas, basta comparar su embedding con los embeddings de las frases "una foto de un {gato}", "una foto de un {perro}", etc., y quedarse con la más cercana — **sin entrenar** un clasificador.

In [ ]:
# Demo: clasificación zero-shot estilo CLIP (en un espacio de embeddings de juguete)
# Imaginemos vectores ya alineados imagen<->texto en un espacio común
torch.manual_seed(1)
def norm(v): return v / v.norm(dim=-1, keepdim=True)
textos = {"un gato": torch.tensor([1.,0.,0.]),
          "un perro": torch.tensor([0.,1.,0.]),
          "un auto": torch.tensor([0.,0.,1.])}
# embedding de una imagen "parecida a gato" (con algo de ruido)
img_emb = norm(torch.tensor([0.9, 0.1, 0.05]) + 0.05*torch.randn(3))
print("Similitud coseno imagen <-> cada texto:")
for t, v in textos.items():
    sim = torch.dot(img_emb, norm(v)).item()
    print(f"  '{t}': {sim:.3f}")
mejor = max(textos, key=lambda t: torch.dot(img_emb, norm(textos[t])))
print("\n-> Predicción zero-shot:", mejor)

### BLIP-2 y los LLM con visión
**BLIP-2** conecta un encoder visual congelado con un LLM congelado mediante un módulo puente ligero y entrenable (el **Q-Former**), que selecciona las características visuales relevantes y las traduce al "idioma" del LLM. Permite *image captioning* y *visual question answering* con poco entrenamiento.

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/m4_06_blip2.png" width="680">

<sub>BLIP-2: un Q-Former entrenable conecta un encoder de imagen congelado con un LLM congelado.</sub>

**LLaVA** lleva esto al paradigma de los LLM: proyecta las características de imagen (de un encoder CLIP) al espacio de *tokens* de un LLM, de modo que el modelo "ve" la imagen como si fueran tokens más. Así nacen los **asistentes multimodales** que conversan sobre imágenes.

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/m4_07_llava.png" width="760">

<sub>LLaVA: una proyección mapea los embeddings visuales al espacio de tokens del LLM; la imagen entra como 'tokens visuales'.</sub>

## Bloque 4 · Agentes

Un LLM por sí solo solo predice texto. Un **agente** usa al LLM como "cerebro" que **razona** y **decide acciones**, ejecutándolas con **herramientas** externas (búsqueda, código, APIs, calculadora). Esto supera dos límites del LLM: conocimiento desactualizado y la tendencia a inventar (*alucinar*).

### El patrón ReAct (Reasoning + Acting) [Ver infografía](https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/infografias/M4/05_React.png)
**ReAct** intercala razonamiento y acción en un bucle: el LLM produce un **Thought** (qué hacer), una **Action** (llamar una herramienta), recibe una **Observation** (el resultado), y repite hasta dar una respuesta final. Las acciones traen información real; el razonamiento decide los siguientes pasos.

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/m4_08_agente_react.png" width="680">

<sub>Bucle ReAct: el LLM alterna Thought → Action (herramienta) → Observation, hasta producir la respuesta final.</sub>

In [ ]:
# Demo: simulación del bucle ReAct con una herramienta de juguete (sin LLM real)
def calculadora(expr):           # una "herramienta"
    return eval(expr, {"__builtins__": {}})

# Traza que un agente ReAct produciría para: "¿Cuánto es el 15% de 240, más 10?"
traza = [
    ("Thought", "Necesito calcular el 15% de 240."),
    ("Action",  "calculadora('240 * 0.15')"),
    ("Observation", str(calculadora('240 * 0.15'))),
    ("Thought", "Ahora le sumo 10."),
    ("Action",  "calculadora('36 + 10')"),
    ("Observation", str(calculadora('36 + 10'))),
    ("Final Answer", "El resultado es 46."),
]
for paso, contenido in traza:
    print(f"{paso:13s}: {contenido}")

### RAG (Retrieval-Augmented Generation) [Ver infografía](https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/infografias/M4/06_RAG.png)
**RAG** le da al LLM acceso a una base de conocimiento externa. En vez de reentrenar el modelo, se **recuperan** documentos relevantes y se inyectan en el prompt:

1. **Indexación (offline):** los documentos se parten en *chunks*, se convierten en vectores con un *embedding model* y se guardan en una *vector database*.
2. **Consulta (online):** la pregunta se convierte en vector, se buscan los *chunks* más similares (similitud coseno) y se añaden al prompt.
3. **Generación:** el LLM responde **fundamentado** en esos documentos.

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/m4_09_rag.png" width="700">

<sub>RAG: indexación de documentos en una base vectorial y, ante una consulta, recuperación de los chunks relevantes para fundamentar la respuesta del LLM.</sub>

In [ ]:
# Demo: un RAG mínimo con embeddings de bolsa-de-palabras y similitud coseno
corpus = [
    "El perceptrón multicapa se entrena con backpropagation.",
    "Las redes convolucionales explotan la estructura espacial de las imágenes.",
    "Los Transformers usan el mecanismo de self-attention.",
    "Stable Diffusion genera imágenes mediante un proceso de difusión.",
]
# Quitamos palabras vacías (stopwords) para que pesen los términos con contenido
STOP = set("el la los las un una de del con se que en y a por para es son".split())
def tokenizar(t):
    t = t.lower().replace(".", "").replace("¿", "").replace("?", "")
    return [w for w in t.split() if w not in STOP]

vocab = sorted(set(w for doc in corpus for w in tokenizar(doc)))
def embed(texto):
    v = np.zeros(len(vocab))
    for w in tokenizar(texto):
        if w in vocab: v[vocab.index(w)] += 1
    n = np.linalg.norm(v); return v/n if n>0 else v
emb_corpus = np.stack([embed(d) for d in corpus])

pregunta = "¿Cómo se entrena un perceptrón con backpropagation?"
sims = emb_corpus @ embed(pregunta)          # similitud coseno (vectores normalizados)
top = sims.argmax()
print("Pregunta:", pregunta)
print("\nChunk recuperado (más similar):")
print(" ->", corpus[top], f"(sim={sims[top]:.2f})")
print("\nEse chunk se inyectaría en el prompt del LLM para fundamentar la respuesta.")

### Límites de los agentes
Los agentes son potentes pero frágiles: pueden encadenar errores, hacer llamadas innecesarias, ser costosos y lentos, y plantean riesgos de seguridad (ejecutar acciones reales). Es un área de investigación muy activa.

## 🔬 Fronteras de la investigación: SSL, DINO, SAM y World Models

Para cerrar, miremos algunas de las direcciones más activas hoy. Todas comparten un hilo: **aprender de datos sin (o con muy pocas) etiquetas**, el ingrediente que hizo posibles los modelos fundacionales.

### Aprendizaje auto-supervisado (self-supervised learning, SSL) [Ver infografía](https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/infografias/M4/07_SSL.png)

Etiquetar datos es caro; datos sin etiquetar hay infinitos. El **SSL** aprende creando una *tarea pretexto* cuyas "etiquetas" salen de los propios datos. Ya vimos ejemplos: BERT predice palabras enmascaradas; los LLM predicen el siguiente token; CLIP empareja imágenes y texto. Son las tres grandes familias de SSL:

- **Predicción enmascarada:** ocultar parte de la entrada y reconstruirla (BERT, Masked Autoencoders).
- **Contrastiva:** acercar representaciones de vistas relacionadas y alejar las no relacionadas (SimCLR, CLIP).
- **Auto-destilación:** una red aprende a imitar a otra versión de sí misma sin etiquetas (DINO).

El SSL es lo que permite preentrenar **modelos fundacionales** sobre datos masivos y luego adaptarlos a muchas tareas.

### DINO / DINOv2: ViT auto-supervisados [Ver infografía](https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/infografias/M4/08_DINO.png)

**DINO** (*self-DIstillation with NO labels*, Meta 2021) entrena un Vision Transformer **sin etiquetas** con un esquema *student–teacher*: ambos comparten arquitectura, pero el **teacher** no se entrena por gradiente, sino que es un **promedio móvil exponencial (EMA)** de los pesos del student. A cada uno se le muestran *vistas* (crops) distintas de la misma imagen, y el student aprende a reproducir la distribución de salida del teacher. No usa etiquetas ni pares negativos.

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/m4_10_dino.png" width="680">

<sub>DINO: el student y el teacher (EMA del student) ven distintos crops de la misma imagen; el student aprende a imitar al teacher, sin etiquetas.</sub>

Una **propiedad emergente** notable: los mapas de atención de un ViT entrenado con DINO **segmentan objetos** sin que nadie se lo enseñara. **DINOv2** (Meta 2023) escaló la idea a ~142 millones de imágenes curadas, produciendo un **modelo fundacional de visión**: sus *features* sirven directamente —sin fine-tuning— para clasificación, segmentación y estimación de profundidad.

In [ ]:
# Demo: la actualización EMA del teacher en DINO
import torch
torch.manual_seed(0)
lam = 0.99                              # factor del promedio móvil
theta_student = torch.randn(5)
theta_teacher = theta_student.clone()
for step in range(5):
    theta_student = theta_student + 0.3*torch.randn(5)              # el student "aprende"
    theta_teacher = lam*theta_teacher + (1-lam)*theta_student       # teacher = EMA del student
    print(f"paso {step}: ||teacher - student|| = {(theta_teacher-theta_student).norm():.3f}")
print("\nEl teacher sigue de forma lenta y estable al student (no se entrena directamente).")

### Segment Anything (SAM): un modelo fundacional para segmentar [Ver infografía](https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/infografias/M4/09_SAM.png)

**SAM** (Meta, abril 2023) llevó la idea de modelo fundacional a la **segmentación**. Es *promptable*: dado un *prompt* (un punto, una caja, una máscara aproximada), devuelve la máscara del objeto correspondiente, **sin fine-tuning** y sobre imágenes nunca vistas (*zero-shot*).

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/m4_11_sam.png" width="720">

<sub>SAM: un image encoder pesado (ViT) y un prompt encoder ligero alimentan un mask decoder ligero que produce la máscara.</sub>

Su arquitectura separa el cómputo: un **image encoder** pesado (ViT) que se ejecuta una vez por imagen, un **prompt encoder** ligero, y un **mask decoder** ligero que permite interactividad en tiempo real. Se entrenó sobre **SA-1B** (1.100 millones de máscaras en 11 millones de imágenes), generadas en un bucle modelo-humano.

La familia siguió evolucionando: **SAM 2** (2024) unifica imagen y **video** con un módulo de **memoria** que sigue objetos a través de los frames; y **SAM 3** (2025) incorpora *prompts* de **texto y de concepto** ("segmenta todos los gatos atigrados").

### World Models y JEPA: hacia agentes que entienden el mundo físico [Ver infografía](https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/infografias/M4/10_JEPA.png)

Un LLM predice texto, pero para **actuar en el mundo físico** un agente necesita un modelo interno de cómo evoluciona el entorno: un **world model**. Con él, el agente puede **entender, predecir y planificar** ("pensar antes de actuar").

La apuesta de Yann LeCun (Meta) es **JEPA** (*Joint Embedding Predictive Architecture*, 2022). La diferencia clave con los modelos generativos: en vez de predecir **píxeles**, JEPA predice **representaciones** (en el espacio latente). Predecir cada píxel obliga a modelar detalles irrelevantes e impredecibles; predecir representaciones se concentra en la estructura que importa.

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/m4_12_jepa.png" width="700">

<sub>JEPA: un context encoder y un predictor estiman la representación de la parte oculta, que se compara —en el espacio latente— con la del target encoder (un EMA).</sub>

La arquitectura usa un **context encoder** (parte visible), un **target encoder** (un EMA, produce la representación objetivo) y un **predictor**; el entrenamiento minimiza la distancia **en el espacio de representaciones**.

La línea de modelos:
- **I-JEPA** (imágenes, 2023): predice representaciones de regiones ocultas de una imagen.
- **V-JEPA** (video, feb. 2024): lleva la idea al video.
- **V-JEPA 2** (jun. 2025): un world model de **1.200 millones de parámetros**, preentrenado con **más de 1 millón de horas** de video de internet. Logra resultados de punta en comprensión del movimiento y anticipación de acciones, y —con muy pocos datos de robots (~62 h)— habilita **planificación robótica zero-shot** en entornos nuevos. Es un paso hacia lo que Meta llama *advanced machine intelligence*.

In [ ]:
# Demo: JEPA predice REPRESENTACIONES, no píxeles
import torch, torch.nn.functional as F
torch.manual_seed(0)
contexto_emb     = torch.randn(8)          # representación del contexto (parte visible)
objetivo_emb     = torch.randn(8)          # representación objetivo (del target encoder)
predictor        = torch.nn.Linear(8, 8)   # estima la representación objetivo
pred             = predictor(contexto_emb)
loss_latente = F.mse_loss(pred, objetivo_emb)
print("Loss de JEPA = distancia en el espacio LATENTE:", round(loss_latente.item(), 3))
print("Los modelos generativos, en cambio, miden el error PIXEL a PIXEL en el espacio de la imagen.")
print("Predecir en latente evita malgastar capacidad en detalles impredecibles.")

## Bloque 5 · Cierre del curso

En 8 sesiones recorrimos el deep learning de punta a punta:

- **Módulo 1 — Fundamentos:** del perceptrón al MLP, pérdidas, gradiente descendente, backpropagation y optimización. *La receta universal.*
- **Módulo 2 — Visión:** convolución, CNN, arquitecturas (ResNet), transfer learning y Vision Transformers.
- **Módulo 3 — NLP:** embeddings, RNN/LSTM, atención, Transformers y LLMs.
- **Módulo 4 — Avanzado:** modelos generativos, difusión, multimodal y agentes.

Una idea atraviesa todo el curso: **aprender representaciones a partir de datos, optimizando una pérdida con gradiente descendente.** Cambian los datos, las arquitecturas y los objetivos, pero el principio se mantiene.

¿Hacia dónde sigue el campo? Modelos cada vez más generales y multimodales, agentes más confiables, eficiencia (modelos más pequeños y rápidos), y un foco creciente en seguridad, alineamiento e interpretabilidad.

**En la clase práctica (Clase 8)** cerramos con las herramientas de hoy: clasificación zero-shot con CLIP, generación de imágenes con difusión, y un sistema RAG.